# SupplyShield AI — System Integration & Unified Risk Layer

This notebook integrates the SupplyShield AI supply-chain intelligence
pipeline with the WebShield intelligence layer.

The integration layer:

1. Loads supply-chain risk outputs.
2. Loads WebShield intelligence outputs.
3. Aligns records using robust entity keys.
4. Generates a unified SupplyShield risk score.
5. Produces explainable risk factors.
6. Updates the SQL database.
7. Creates dashboard-ready views.
8. Validates the complete integrated pipeline.

This notebook is an integration and validation layer rather than a new ML
training stage.

In [1]:
# ============================================================
# CELL 2 — IMPORTS & PROJECT CONFIGURATION
# ============================================================

from __future__ import annotations

import json
import re
import math
import sqlite3
import hashlib
import logging
import warnings

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("SupplyShield-Integration")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR

for candidate in [CURRENT_DIR] + list(CURRENT_DIR.parents):

    if (candidate / "member2").exists():

        PROJECT_ROOT = candidate
        break

# ------------------------------------------------------------
# Important directories
# ------------------------------------------------------------

MEMBER2_DIR = PROJECT_ROOT / "member2"

DATA_DIR = MEMBER2_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

WEB_SHIELD_DIR = (
    MEMBER2_DIR
    / "outputs"
    / "webshield"
)

INTEGRATION_DIR = (
    MEMBER2_DIR
    / "outputs"
    / "integration"
)

DATABASE_DIR = (
    MEMBER2_DIR
    / "database"
)

INTEGRATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------

SUPPLY_RISK_JSON = (
    PROCESSED_DIR
    / "risk_scoring"
    / "final_risk_scored_supply_data.json"
)

SUPPLY_RISK_CSV = (
    PROCESSED_DIR
    / "risk_scoring"
    / "final_risk_scored_supply_data.csv"
)

WEB_SHIELD_JSON = (
    WEB_SHIELD_DIR
    / "webshield_risk_results.json"
)

WEB_SHIELD_CSV = (
    WEB_SHIELD_DIR
    / "webshield_risk_results.csv"
)

DATABASE_PATH = (
    DATABASE_DIR
    / "supplyshield.db"
)

print("=" * 85)
print("SUPPLYSHIELD AI — SYSTEM INTEGRATION")
print("=" * 85)

print(f"Project root       : {PROJECT_ROOT}")
print(f"Supply risk JSON   : {SUPPLY_RISK_JSON}")
print(f"WebShield JSON     : {WEB_SHIELD_JSON}")
print(f"Database           : {DATABASE_PATH}")
print(f"Integration output : {INTEGRATION_DIR}")

print("=" * 85)

SUPPLYSHIELD AI — SYSTEM INTEGRATION
Project root       : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI
Supply risk JSON   : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\final_risk_scored_supply_data.json
WebShield JSON     : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\webshield\webshield_risk_results.json
Database           : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\database\supplyshield.db
Integration output : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\integration


## 1. Load Upstream Intelligence

Notebook 05 provides the supply-chain risk intelligence.

Notebook 07 provides WebShield intelligence.

Both outputs are treated as upstream analytical products and are integrated
without retraining their underlying models.

In [2]:
# ============================================================
# CELL 4 — ROBUST JSON / CSV LOADER
# ============================================================

def load_structured_dataset(
    json_path: Path,
    csv_path: Path,
    dataset_name: str
) -> pd.DataFrame:
    """
    Load JSON as the preferred source and CSV as fallback.

    Supports:
    - list of records
    - dictionary containing records/data/items/results
    - single dictionary record
    """

    if json_path.exists():

        logger.info(
            "Loading %s from JSON.",
            dataset_name
        )

        with open(
            json_path,
            "r",
            encoding="utf-8"
        ) as file:

            payload = json.load(file)

        if isinstance(payload, list):

            records = payload

        elif isinstance(payload, dict):

            records = None

            for key in [
                "records",
                "data",
                "items",
                "results",
                "products"
            ]:

                if (
                    key in payload
                    and isinstance(payload[key], list)
                ):

                    records = payload[key]
                    break

            if records is None:
                records = [payload]

        else:

            raise ValueError(
                f"Unsupported {dataset_name} JSON structure."
            )

        dataframe = pd.DataFrame(records)

    elif csv_path.exists():

        logger.info(
            "JSON unavailable. Loading %s from CSV.",
            dataset_name
        )

        dataframe = pd.read_csv(
            csv_path
        )

    else:

        raise FileNotFoundError(
            f"\n{dataset_name} output not found.\n\n"
            f"Expected:\n{json_path}\n"
            f"or:\n{csv_path}\n\n"
            "Complete the previous notebook first."
        )

    if dataframe.empty:

        raise ValueError(
            f"{dataset_name} dataset is empty."
        )

    return dataframe


supply_df = load_structured_dataset(
    SUPPLY_RISK_JSON,
    SUPPLY_RISK_CSV,
    "Supply-chain risk"
)

webshield_df = load_structured_dataset(
    WEB_SHIELD_JSON,
    WEB_SHIELD_CSV,
    "WebShield"
)


print("=" * 85)
print("UPSTREAM DATASETS LOADED")
print("=" * 85)

print(
    f"Supply-chain records : {len(supply_df):,}"
)

print(
    f"WebShield records    : {len(webshield_df):,}"
)

print(
    f"Supply columns       : {len(supply_df.columns):,}"
)

print(
    f"WebShield columns    : {len(webshield_df.columns):,}"
)

2026-08-21 10:38:22,964 | INFO | Loading Supply-chain risk from JSON.
2026-08-21 10:38:23,094 | INFO | Loading WebShield from JSON.


UPSTREAM DATASETS LOADED
Supply-chain records : 303
WebShield records    : 303
Supply columns       : 168
WebShield columns    : 28


In [3]:
# ============================================================
# CELL 5 — COLUMN NORMALIZATION
# ============================================================

def normalize_column_name(
    column: Any
) -> str:

    text = str(column).strip().lower()

    text = re.sub(
        r"[^a-z0-9]+",
        "_",
        text
    )

    text = re.sub(
        r"_+",
        "_",
        text
    )

    return text.strip("_")


supply_df.columns = [
    normalize_column_name(column)
    for column in supply_df.columns
]

webshield_df.columns = [
    normalize_column_name(column)
    for column in webshield_df.columns
]


print("=" * 85)
print("NORMALIZED COLUMN NAMES")
print("=" * 85)

print("\nSupply-chain columns:")
print(list(supply_df.columns))

print("\nWebShield columns:")
print(list(webshield_df.columns))

NORMALIZED COLUMN NAMES

Supply-chain columns:
['record_id', 'source', 'title', 'company', 'supplier', 'product', 'event', 'location', 'currency', 'url', 'availability_state', 'supplier_normalized', 'source_normalized', 'preliminary_risk_band', 'price', 'price_log', 'price_deviation_pct', 'price_robust_zscore', 'price_zscore', 'price_percentile', 'price_iqr_outlier', 'product_price_deviation_pct', 'supplier_price_deviation_pct', 'availability_risk_score', 'availability_text_length', 'availability_missing_flag', 'rating', 'rating_normalized', 'rating_risk_score', 'review_length', 'review_word_count', 'review_char_count', 'review_exclamation_count', 'review_question_count', 'review_uppercase_ratio', 'review_missing_flag', 'rating_missing_flag', 'review_quality_signal', 'disruption_keyword_count', 'negative_keyword_count', 'urgency_keyword_count', 'counterfeit_keyword_count', 'disruption_signal_flag', 'negative_signal_flag', 'urgency_signal_flag', 'counterfeit_signal_flag', 'text_risk_sco

In [4]:
# ============================================================
# CELL 6 — INTEGRATION UTILITY FUNCTIONS
# ============================================================

def safe_text(
    value: Any
) -> str:

    if value is None:
        return ""

    if isinstance(
        value,
        float
    ) and np.isnan(value):

        return ""

    return str(value).strip()


def safe_float(
    value: Any,
    default: float = np.nan
) -> float:

    if value is None:
        return default

    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):

        if pd.isna(value):
            return default

        return float(value)

    text = safe_text(value)

    if not text:
        return default

    text = text.replace(",", "")

    match = re.search(
        r"-?\d+(?:\.\d+)?",
        text
    )

    if not match:
        return default

    try:

        return float(
            match.group()
        )

    except ValueError:

        return default


def normalize_identifier(
    value: Any
) -> str:
    """
    Create a deterministic entity matching key.

    Example:
        '  ABC Components Ltd. '
        →
        'abccomponentsltd'
    """

    text = safe_text(value).lower()

    text = re.sub(
        r"[^a-z0-9]+",
        "",
        text
    )

    return text


def clip_score(
    value: Any
) -> float:

    score = safe_float(
        value,
        0.0
    )

    return float(
        np.clip(
            score,
            0,
            100
        )
    )


def score_to_band(
    score: float
) -> str:

    score = clip_score(
        score,
        0.0
    )

    if score >= 80:
        return "CRITICAL"

    if score >= 60:
        return "HIGH"

    if score >= 40:
        return "MEDIUM"

    if score >= 20:
        return "LOW"

    return "MINIMAL"


def stable_hash(
    values: List[Any]
) -> str:

    normalized = "|".join(
        safe_text(value).lower()
        for value in values
    )

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


logger.info(
    "Integration utility functions initialized."
)

2026-08-21 10:38:50,306 | INFO | Integration utility functions initialized.


## 2. Prepare Supply-Chain Risk Data

The integration layer expects the final score from Notebook 05.

Because the exact upstream column names may evolve during development, the
implementation detects compatible aliases rather than relying on a single
hard-coded schema.

In [5]:
# ============================================================
# CELL 8 — SUPPLY-RISK COLUMN RESOLUTION
# ============================================================

def resolve_column(
    dataframe: pd.DataFrame,
    candidates: List[str]
) -> Optional[str]:

    normalized = {
        normalize_column_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:

        candidate_normalized = normalize_column_name(
            candidate
        )

        if candidate_normalized in normalized:

            return normalized[
                candidate_normalized
            ]

    return None


SUPPLY_COLUMNS = {

    "supplier": resolve_column(
        supply_df,
        [
            "supplier",
            "supplier_name",
            "seller",
            "vendor"
        ]
    ),

    "company": resolve_column(
        supply_df,
        [
            "company",
            "company_name",
            "manufacturer",
            "brand"
        ]
    ),

    "product": resolve_column(
        supply_df,
        [
            "product",
            "product_name",
            "item"
        ]
    ),

    "title": resolve_column(
        supply_df,
        [
            "title",
            "headline"
        ]
    ),

    "source": resolve_column(
        supply_df,
        [
            "source",
            "platform",
            "website"
        ]
    ),

    "location": resolve_column(
        supply_df,
        [
            "location",
            "country",
            "region"
        ]
    ),

    "final_risk": resolve_column(
        supply_df,
        [
            "final_risk_score",
            "overall_risk_score",
            "risk_score"
        ]
    ),

    "risk_band": resolve_column(
        supply_df,
        [
            "risk_band",
            "final_risk_band"
        ]
    ),

    "risk_reasons": resolve_column(
        supply_df,
        [
            "risk_reasons_text",
            "risk_reasons",
            "reasons"
        ]
    ),

    "nlp_risk": resolve_column(
        supply_df,
        [
            "nlp_risk_score",
            "risk_nlp_risk",
            "nlp_risk"
        ]
    ),

    "supplier_risk": resolve_column(
        supply_df,
        [
            "supplier_risk_score",
            "risk_supplier_risk",
            "supplier_risk"
        ]
    ),

    "anomaly_risk": resolve_column(
        supply_df,
        [
            "anomaly_risk_score",
            "risk_anomaly_risk",
            "anomaly_risk"
        ]
    ),

    "disruption_risk": resolve_column(
        supply_df,
        [
            "disruption_risk_score",
            "risk_disruption_risk",
            "disruption_risk"
        ]
    ),

    "market_risk": resolve_column(
        supply_df,
        [
            "market_risk_score",
            "risk_market_risk",
            "market_risk"
        ]
    )
}


print("=" * 85)
print("SUPPLY-RISK COLUMN RESOLUTION")
print("=" * 85)

for logical_name, actual_column in SUPPLY_COLUMNS.items():

    print(
        f"{logical_name:<20} : "
        f"{actual_column or 'NOT FOUND'}"
    )

SUPPLY-RISK COLUMN RESOLUTION
supplier             : supplier
company              : company
product              : product
title                : title
source               : source
location             : location
final_risk           : final_risk_score
risk_band            : risk_band
risk_reasons         : risk_reasons_text
nlp_risk             : nlp_risk_score
supplier_risk        : risk_supplier_risk
anomaly_risk         : risk_anomaly_risk
disruption_risk      : risk_disruption_risk
market_risk          : risk_market_risk


In [6]:
# ============================================================
# CELL 9 — PREPARE CANONICAL SUPPLY-RISK DATASET
# ============================================================

integrated_supply = pd.DataFrame(
    index=supply_df.index
)


def copy_resolved_column(
    target_name: str,
    source_column: Optional[str],
    default: Any = ""
):

    if source_column:

        integrated_supply[target_name] = (
            supply_df[source_column]
        )

    else:

        integrated_supply[target_name] = default


copy_resolved_column(
    "supplier",
    SUPPLY_COLUMNS["supplier"]
)

copy_resolved_column(
    "company",
    SUPPLY_COLUMNS["company"]
)

copy_resolved_column(
    "product",
    SUPPLY_COLUMNS["product"]
)

copy_resolved_column(
    "title",
    SUPPLY_COLUMNS["title"]
)

copy_resolved_column(
    "source",
    SUPPLY_COLUMNS["source"]
)

copy_resolved_column(
    "location",
    SUPPLY_COLUMNS["location"]
)

copy_resolved_column(
    "supply_chain_risk",
    SUPPLY_COLUMNS["final_risk"],
    0.0
)

copy_resolved_column(
    "supply_chain_risk_band",
    SUPPLY_COLUMNS["risk_band"],
    ""
)

copy_resolved_column(
    "supply_chain_reasons",
    SUPPLY_COLUMNS["risk_reasons"],
    ""
)

copy_resolved_column(
    "nlp_risk",
    SUPPLY_COLUMNS["nlp_risk"],
    0.0
)

copy_resolved_column(
    "supplier_risk",
    SUPPLY_COLUMNS["supplier_risk"],
    0.0
)

copy_resolved_column(
    "anomaly_risk",
    SUPPLY_COLUMNS["anomaly_risk"],
    0.0
)

copy_resolved_column(
    "disruption_risk",
    SUPPLY_COLUMNS["disruption_risk"],
    0.0
)

copy_resolved_column(
    "market_risk",
    SUPPLY_COLUMNS["market_risk"],
    0.0
)


# ------------------------------------------------------------
# Clean text fields
# ------------------------------------------------------------

for column in [
    "supplier",
    "company",
    "product",
    "title",
    "source",
    "location"
]:

    integrated_supply[column] = (
        integrated_supply[column]
        .apply(safe_text)
    )


# ------------------------------------------------------------
# Clean numeric fields
# ------------------------------------------------------------

for column in [
    "supply_chain_risk",
    "nlp_risk",
    "supplier_risk",
    "anomaly_risk",
    "disruption_risk",
    "market_risk"
]:

    integrated_supply[column] = (
        integrated_supply[column]
        .apply(
            lambda value: clip_score(
                value
            )
        )
    )


# ------------------------------------------------------------
# Generate entity keys
# ------------------------------------------------------------

integrated_supply["supplier_key"] = (
    integrated_supply["supplier"]
    .apply(normalize_identifier)
)

integrated_supply["company_key"] = (
    integrated_supply["company"]
    .apply(normalize_identifier)
)

integrated_supply["product_key"] = (
    integrated_supply["product"]
    .apply(normalize_identifier)
)

integrated_supply["title_key"] = (
    integrated_supply["title"]
    .apply(normalize_identifier)
)


print("=" * 85)
print("SUPPLY-RISK DATA PREPARED")
print("=" * 85)

print(
    f"Records: {len(integrated_supply):,}"
)

display(
    integrated_supply.head(10)
)

SUPPLY-RISK DATA PREPARED
Records: 303


,supplier,company,product,title,source,location,supply_chain_risk,supply_chain_risk_band,supply_chain_reasons,nlp_risk,supplier_risk,anomaly_risk,disruption_risk,market_risk,supplier_key,company_key,product_key,title_key
0,,GlimmerHome,Wall Mount Mop Holder – No-Slide Grip for Home...,Wall Mount Mop Holder – No-Slide Grip for Home...,DeoDap,,8.28,LOW,Unusual/anomalous behaviour detected (72.8/100),0.050000,0.0,72.830795,0.0,0.0,,glimmerhome,wallmountmopholdernoslidegripforhomegarage,wallmountmopholdernoslidegripforhomegarageglim...
1,,DeoDap,Compact TianMu Tool Set – Essential 9-Piece Re...,Compact TianMu Tool Set – Essential 9-Piece Re...,DeoDap,,8.37,LOW,Unusual/anomalous behaviour detected (73.7/100),0.050000,0.0,73.741645,0.0,0.0,,deodap,compacttianmutoolsetessential9piecerepairkitfo...,compacttianmutoolsetessential9piecerepairkitfo...
2,,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),Manual Wall Fastening Nail Gun Tool Set (1 Set),DeoDap,,9.66,LOW,Unusual/anomalous behaviour detected (80.7/100),0.079295,0.0,80.741588,0.0,0.0,,deodap,manualwallfasteningnailguntoolset1set,manualwallfasteningnailguntoolset1set
3,,DeoDap,Mini Precision Screwdriver Set – Compact & Mul...,Mini Precision Screwdriver Set – Compact & Mul...,DeoDap,,8.04,LOW,Unusual/anomalous behaviour detected (72.0/100),0.042345,0.0,71.977422,0.0,0.0,,deodap,miniprecisionscrewdriversetcompactmultipurpose...,miniprecisionscrewdriversetcompactmultipurpose...
4,,DeoDap,Electric Drill Machine – Compact & Powerful 28...,Electric Drill Machine – Compact & Powerful 28...,DeoDap,,8.82,LOW,Unusual/anomalous behaviour detected (85.0/100),0.015960,0.0,85.001131,0.0,0.0,,deodap,electricdrillmachinecompactpowerful280wforvers...,electricdrillmachinecompactpowerful280wforvers...
5,,BoltForce,Leak Proof Tape – Instant Waterproof Seal for ...,Leak Proof Tape – Instant Waterproof Seal for ...,DeoDap,,11.45,LOW,Unusual/anomalous behaviour detected (98.7/100),0.079295,0.0,98.687571,0.0,0.0,,boltforce,leakprooftapeinstantwaterproofsealforrepairs,leakprooftapeinstantwaterproofsealforrepairsbo...
6,,DeoDap,Metal Try Square Ruler Set – Durable 2-Piece P...,Metal Try Square Ruler Set – Durable 2-Piece P...,DeoDap,,7.61,LOW,Unusual/anomalous behaviour detected (66.1/100),0.050000,0.0,66.133028,0.0,0.0,,deodap,metaltrysquarerulersetdurable2pieceprecisiontool,metaltrysquarerulersetdurable2pieceprecisiontool
7,,DeoDap,Heavy Duty Sledge Hammer Rubber Mallet for Con...,Heavy Duty Sledge Hammer Rubber Mallet for Con...,DeoDap,,7.95,LOW,Unusual/anomalous behaviour detected (69.5/100),0.050000,0.0,69.540926,0.0,0.0,,deodap,heavydutysledgehammerrubbermalletforconstructi...,heavydutysledgehammerrubbermalletforconstructi...
8,,BoltForce,Hardware Tool Set – 11 Pcs Multi-Functional Kit,Hardware Tool Set – 11 Pcs Multi-Functional Ki...,DeoDap,,9.48,LOW,Unusual/anomalous behaviour detected (84.8/100),0.050000,0.0,84.803158,0.0,0.0,,boltforce,hardwaretoolset11pcsmultifunctionalkit,hardwaretoolset11pcsmultifunctionalkitboltforce
9,,DeoDap,Heavy Duty Electric Drill – Powerful & Versati...,Heavy Duty Electric Drill – Powerful & Versati...,DeoDap,,7.58,LOW,Unusual/anomalous behaviour detected (72.6/100),0.015960,0.0,72.559790,0.0,0.0,,deodap,heavydutyelectricdrillpowerfulversatileforprof...,heavydutyelectricdrillpowerfulversatileforprof...


## 3. Prepare WebShield Intelligence

WebShield contributes independent web-risk signals including suspicious
pricing, seller/reputation risk, review anomalies, listing quality, product
similarity, and counterfeit-risk proxy scores.

In [7]:
# ============================================================
# CELL 11 — WEBSHIELD COLUMN RESOLUTION
# ============================================================

WEBSHIELD_COLUMNS = {

    "supplier": resolve_column(
        webshield_df,
        [
            "web_seller",
            "supplier",
            "seller",
            "vendor"
        ]
    ),

    "company": resolve_column(
        webshield_df,
        [
            "company",
            "company_name",
            "brand"
        ]
    ),

    "product": resolve_column(
        webshield_df,
        [
            "product",
            "product_name"
        ]
    ),

    "title": resolve_column(
        webshield_df,
        [
            "title",
            "product_title"
        ]
    ),

    "source": resolve_column(
        webshield_df,
        [
            "source",
            "platform"
        ]
    ),

    "url": resolve_column(
        webshield_df,
        [
            "url",
            "product_url",
            "link"
        ]
    ),

    "webshield_risk": resolve_column(
        webshield_df,
        [
            "webshield_risk_score_100",
            "webshield_risk_score"
        ]
    ),

    "counterfeit_risk": resolve_column(
        webshield_df,
        [
            "web_counterfeit_risk_100",
            "web_counterfeit_risk_score"
        ]
    ),

    "price_anomaly": resolve_column(
        webshield_df,
        [
            "web_price_anomaly_score"
        ]
    ),

    "seller_risk": resolve_column(
        webshield_df,
        [
            "web_seller_reputation_score"
        ]
    ),

    "review_anomaly": resolve_column(
        webshield_df,
        [
            "web_review_anomaly_score"
        ]
    ),

    "similarity_risk": resolve_column(
        webshield_df,
        [
            "web_product_similarity_risk"
        ]
    ),

    "listing_quality": resolve_column(
        webshield_df,
        [
            "web_listing_quality_risk"
        ]
    ),

    "action": resolve_column(
        webshield_df,
        [
            "webshield_action"
        ]
    ),

    "reasons": resolve_column(
        webshield_df,
        [
            "webshield_risk_reasons"
        ]
    )
}


print("=" * 85)
print("WEBSHIELD COLUMN RESOLUTION")
print("=" * 85)

for logical_name, actual_column in WEBSHIELD_COLUMNS.items():

    print(
        f"{logical_name:<20} : "
        f"{actual_column or 'NOT FOUND'}"
    )

WEBSHIELD COLUMN RESOLUTION
supplier             : web_seller
company              : company
product              : product
title                : title
source               : source
url                  : url
webshield_risk       : webshield_risk_score_100
counterfeit_risk     : web_counterfeit_risk_100
price_anomaly        : web_price_anomaly_score
seller_risk          : web_seller_reputation_score
review_anomaly       : web_review_anomaly_score
similarity_risk      : web_product_similarity_risk
listing_quality      : web_listing_quality_risk
action               : webshield_action
reasons              : webshield_risk_reasons


In [8]:
# ============================================================
# CELL 12 — PREPARE CANONICAL WEBSHIELD DATASET
# ============================================================

integrated_webshield = pd.DataFrame(
    index=webshield_df.index
)


def copy_webshield_column(
    target_name: str,
    source_column: Optional[str],
    default: Any = ""
):

    if source_column:

        integrated_webshield[target_name] = (
            webshield_df[source_column]
        )

    else:

        integrated_webshield[target_name] = default


copy_webshield_column(
    "supplier",
    WEBSHIELD_COLUMNS["supplier"]
)

copy_webshield_column(
    "company",
    WEBSHIELD_COLUMNS["company"]
)

copy_webshield_column(
    "product",
    WEBSHIELD_COLUMNS["product"]
)

copy_webshield_column(
    "title",
    WEBSHIELD_COLUMNS["title"]
)

copy_webshield_column(
    "source",
    WEBSHIELD_COLUMNS["source"]
)

copy_webshield_column(
    "url",
    WEBSHIELD_COLUMNS["url"]
)

copy_webshield_column(
    "webshield_risk",
    WEBSHIELD_COLUMNS["webshield_risk"],
    0.0
)

copy_webshield_column(
    "counterfeit_risk",
    WEBSHIELD_COLUMNS["counterfeit_risk"],
    0.0
)

copy_webshield_column(
    "price_anomaly",
    WEBSHIELD_COLUMNS["price_anomaly"],
    0.0
)

copy_webshield_column(
    "seller_risk",
    WEBSHIELD_COLUMNS["seller_risk"],
    0.0
)

copy_webshield_column(
    "review_anomaly",
    WEBSHIELD_COLUMNS["review_anomaly"],
    0.0
)

copy_webshield_column(
    "similarity_risk",
    WEBSHIELD_COLUMNS["similarity_risk"],
    0.0
)

copy_webshield_column(
    "listing_quality",
    WEBSHIELD_COLUMNS["listing_quality"],
    0.0
)

copy_webshield_column(
    "action",
    WEBSHIELD_COLUMNS["action"],
    "NORMAL"
)

copy_webshield_column(
    "webshield_reasons",
    WEBSHIELD_COLUMNS["reasons"],
    ""
)


# ------------------------------------------------------------
# Clean identity fields
# ------------------------------------------------------------

for column in [
    "supplier",
    "company",
    "product",
    "title",
    "source",
    "url",
    "action"
]:

    integrated_webshield[column] = (
        integrated_webshield[column]
        .apply(safe_text)
    )


# ------------------------------------------------------------
# WebShield scores
#
# WebShield engine stores main scores on 0–100.
# Component signals are normally 0–1.
# ------------------------------------------------------------

for column in [
    "webshield_risk",
    "counterfeit_risk"
]:

    integrated_webshield[column] = (
        integrated_webshield[column]
        .apply(
            lambda value: clip_score(
                value
            )
        )
    )


for column in [
    "price_anomaly",
    "seller_risk",
    "review_anomaly",
    "similarity_risk",
    "listing_quality"
]:

    integrated_webshield[column] = (
        pd.to_numeric(
            integrated_webshield[column],
            errors="coerce"
        )
        .fillna(0)
        .clip(0, 1)
    )


# ------------------------------------------------------------
# Entity keys
# ------------------------------------------------------------

integrated_webshield["supplier_key"] = (
    integrated_webshield["supplier"]
    .apply(normalize_identifier)
)

integrated_webshield["company_key"] = (
    integrated_webshield["company"]
    .apply(normalize_identifier)
)

integrated_webshield["product_key"] = (
    integrated_webshield["product"]
    .apply(normalize_identifier)
)

integrated_webshield["title_key"] = (
    integrated_webshield["title"]
    .apply(normalize_identifier)
)


print("=" * 85)
print("WEBSHIELD DATA PREPARED")
print("=" * 85)

print(
    f"Records: {len(integrated_webshield):,}"
)

display(
    integrated_webshield.head(10)
)

WEBSHIELD DATA PREPARED
Records: 303


,supplier,company,product,title,source,url,webshield_risk,counterfeit_risk,price_anomaly,seller_risk,review_anomaly,similarity_risk,listing_quality,action,webshield_reasons,supplier_key,company_key,product_key,title_key
0,GlimmerHome,GlimmerHome,Wall Mount Mop Holder – No-Slide Grip for Home...,Wall Mount Mop Holder – No-Slide Grip for Home...,DeoDap,https://deodap.in/products/hardware-tool-multi...,15.66,15.27,0.061392,0.3280,0.0,0.098874,0.25,NORMAL,[No strong WebShield anomaly detected],glimmerhome,glimmerhome,wallmountmopholdernoslidegripforhomegarage,wallmountmopholdernoslidegripforhomegarageglim...
1,DeoDap,DeoDap,Compact TianMu Tool Set – Essential 9-Piece Re...,Compact TianMu Tool Set – Essential 9-Piece Re...,DeoDap,https://deodap.in/products/compact-tianmu-comb...,16.76,16.70,0.079600,0.3140,0.0,0.180759,0.25,NORMAL,[No strong WebShield anomaly detected],deodap,deodap,compacttianmutoolsetessential9piecerepairkitfo...,compacttianmutoolsetessential9piecerepairkitfo...
2,DeoDap,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),Manual Wall Fastening Nail Gun Tool Set (1 Set),DeoDap,https://deodap.in/products/manual-wall-fasteni...,25.63,27.27,0.198896,0.3217,0.0,0.634006,0.25,NORMAL,[No strong WebShield anomaly detected],deodap,deodap,manualwallfasteningnailguntoolset1set,manualwallfasteningnailguntoolset1set
3,DeoDap,DeoDap,Mini Precision Screwdriver Set – Compact & Mul...,Mini Precision Screwdriver Set – Compact & Mul...,DeoDap,https://deodap.in/products/mini-precision-scre...,20.05,20.75,0.106653,0.3364,0.0,0.359280,0.25,NORMAL,[No strong WebShield anomaly detected],deodap,deodap,miniprecisionscrewdriversetcompactmultipurpose...,miniprecisionscrewdriversetcompactmultipurpose...
4,DeoDap,DeoDap,Electric Drill Machine – Compact & Powerful 28...,Electric Drill Machine – Compact & Powerful 28...,DeoDap,https://deodap.in/products/high-performance-el...,19.75,20.00,0.129303,0.3203,0.0,0.290732,0.25,NORMAL,[No strong WebShield anomaly detected],deodap,deodap,electricdrillmachinecompactpowerful280wforvers...,electricdrillmachinecompactpowerful280wforvers...
5,BoltForce,BoltForce,Leak Proof Tape – Instant Waterproof Seal for ...,Leak Proof Tape – Instant Waterproof Seal for ...,DeoDap,https://deodap.in/products/0405_leak_proof_tape-1,18.75,18.14,0.113467,0.3777,0.0,0.102985,0.25,NORMAL,[No strong WebShield anomaly detected],boltforce,boltforce,leakprooftapeinstantwaterproofsealforrepairs,leakprooftapeinstantwaterproofsealforrepairsbo...
6,DeoDap,DeoDap,Metal Try Square Ruler Set – Durable 2-Piece P...,Metal Try Square Ruler Set – Durable 2-Piece P...,DeoDap,https://deodap.in/products/heavy-duty-metal-tr...,16.54,16.12,0.087140,0.3203,0.0,0.116642,0.25,NORMAL,[No strong WebShield anomaly detected],deodap,deodap,metaltrysquarerulersetdurable2pieceprecisiontool,metaltrysquarerulersetdurable2pieceprecisiontool
7,DeoDap,DeoDap,Heavy Duty Sledge Hammer Rubber Mallet for Con...,Heavy Duty Sledge Hammer Rubber Mallet for Con...,DeoDap,https://deodap.in/products/heavy-duty-sledge-h...,17.81,17.75,0.100701,0.3189,0.0,0.200152,0.25,NORMAL,[No strong WebShield anomaly detected],deodap,deodap,heavydutysledgehammerrubbermalletforconstructi...,heavydutysledgehammerrubbermalletforconstructi...
8,BoltForce,BoltForce,Hardware Tool Set – 11 Pcs Multi-Functional Kit,Hardware Tool Set – 11 Pcs Multi-Functional Ki...,DeoDap,https://deodap.in/products/15800_multi_hardwar...,18.87,18.80,0.104985,0.3539,0.0,0.203354,0.25,NORMAL,[No strong WebShield anomaly detected],boltforce,boltforce,hardwaretoolset11pcsmultifunctionalkit,hardwaretoolset11pcsmultifunctionalkitboltforce
9,DeoDap,DeoDap,Heavy Duty Electric Drill – Powerful & Versati...,Heavy Duty Electric Drill – Powerful & Versati...,DeoDap,https://deodap.in/products/heavy-duty-electric...,20.15,20.40,0.129303,0.3364,0.0,0.290732,0.25,NORMAL,[No strong WebShield anomaly detected],deodap,deodap,heavydutyelectricdrillpowerfulversatileforprof...,heavydutyelectricdrillpowerfulversatileforprof...


## 4. Entity Resolution

The two intelligence streams may not contain identical text.

The integration layer therefore attempts matching using:

1. Supplier + product
2. Company + product
3. Supplier
4. Product
5. Title

Exact normalized matching is used for reliability. Unmatched records remain
available rather than being incorrectly forced into another entity.

In [9]:
# ============================================================
# CELL 14 — BUILD WEBSHIELD ENTITY INDEX
# ============================================================

def build_lookup(
    dataframe: pd.DataFrame,
    key_columns: List[str]
) -> Dict[str, int]:

    lookup = {}

    for index, row in dataframe.iterrows():

        values = [
            safe_text(row[column])
            for column in key_columns
        ]

        if not any(values):
            continue

        key = "||".join(
            normalize_identifier(value)
            for value in values
        )

        if key.strip("|"):

            # Keep first occurrence for deterministic behaviour.
            if key not in lookup:
                lookup[key] = index

    return lookup


# ------------------------------------------------------------
# Most precise lookup
# ------------------------------------------------------------

lookup_supplier_product = build_lookup(
    integrated_webshield,
    [
        "supplier_key",
        "product_key"
    ]
)


lookup_company_product = build_lookup(
    integrated_webshield,
    [
        "company_key",
        "product_key"
    ]
)


lookup_supplier = build_lookup(
    integrated_webshield,
    [
        "supplier_key"
    ]
)


lookup_product = build_lookup(
    integrated_webshield,
    [
        "product_key"
    ]
)


lookup_title = build_lookup(
    integrated_webshield,
    [
        "title_key"
    ]
)


print("=" * 85)
print("WEBSHIELD ENTITY LOOKUPS CREATED")
print("=" * 85)

print(
    f"Supplier + Product keys : "
    f"{len(lookup_supplier_product):,}"
)

print(
    f"Company + Product keys  : "
    f"{len(lookup_company_product):,}"
)

print(
    f"Supplier keys           : "
    f"{len(lookup_supplier):,}"
)

print(
    f"Product keys            : "
    f"{len(lookup_product):,}"
)

print(
    f"Title keys              : "
    f"{len(lookup_title):,}"
)

WEBSHIELD ENTITY LOOKUPS CREATED
Supplier + Product keys : 303
Company + Product keys  : 303
Supplier keys           : 272
Product keys            : 288
Title keys              : 302


In [10]:
# ============================================================
# CELL 15 — SUPPLY ↔ WEBSHIELD ENTITY MATCHING
# ============================================================

def create_key(
    row: pd.Series,
    columns: List[str]
) -> str:

    values = []

    for column in columns:

        values.append(
            safe_text(
                row.get(column, "")
            )
        )

    return "||".join(
        normalize_identifier(value)
        for value in values
    )


def find_webshield_match(
    row: pd.Series
) -> tuple:

    # --------------------------------------------------------
    # Match 1: supplier + product
    # --------------------------------------------------------

    key = create_key(
        row,
        [
            "supplier_key",
            "product_key"
        ]
    )

    if (
        key.strip("|")
        and key in lookup_supplier_product
    ):

        return (
            lookup_supplier_product[key],
            "SUPPLIER_PRODUCT"
        )


    # --------------------------------------------------------
    # Match 2: company + product
    # --------------------------------------------------------

    key = create_key(
        row,
        [
            "company_key",
            "product_key"
        ]
    )

    if (
        key.strip("|")
        and key in lookup_company_product
    ):

        return (
            lookup_company_product[key],
            "COMPANY_PRODUCT"
        )


    # --------------------------------------------------------
    # Match 3: supplier
    # --------------------------------------------------------

    key = create_key(
        row,
        [
            "supplier_key"
        ]
    )

    if (
        key.strip("|")
        and key in lookup_supplier
    ):

        return (
            lookup_supplier[key],
            "SUPPLIER"
        )


    # --------------------------------------------------------
    # Match 4: product
    # --------------------------------------------------------

    key = create_key(
        row,
        [
            "product_key"
        ]
    )

    if (
        key.strip("|")
        and key in lookup_product
    ):

        return (
            lookup_product[key],
            "PRODUCT"
        )


    # --------------------------------------------------------
    # Match 5: title
    # --------------------------------------------------------

    key = create_key(
        row,
        [
            "title_key"
        ]
    )

    if (
        key.strip("|")
        and key in lookup_title
    ):

        return (
            lookup_title[key],
            "TITLE"
        )


    return (
        None,
        "UNMATCHED"
    )


match_indices = []
match_methods = []


for _, row in integrated_supply.iterrows():

    matched_index, method = (
        find_webshield_match(
            row
        )
    )

    match_indices.append(
        matched_index
    )

    match_methods.append(
        method
    )


integrated_supply["webshield_match_index"] = (
    match_indices
)

integrated_supply["webshield_match_method"] = (
    match_methods
)


match_distribution = (
    integrated_supply[
        "webshield_match_method"
    ]
    .value_counts()
)


print("=" * 85)
print("ENTITY MATCHING RESULTS")
print("=" * 85)

display(
    match_distribution
)

ENTITY MATCHING RESULTS


webshield_match_method
COMPANY_PRODUCT    303
Name: count, dtype: int64

In [12]:
# ============================================================
# CELL 16 — MERGE WEBSHIELD INTELLIGENCE
# Robust version: handles scalar, list and missing values
# ============================================================

WEB_OUTPUT_FIELDS = [
    "webshield_risk",
    "counterfeit_risk",
    "price_anomaly",
    "seller_risk",
    "review_anomaly",
    "similarity_risk",
    "listing_quality",
    "action",
    "webshield_reasons"
]


# ------------------------------------------------------------
# Helper: safely convert any value to a database/dataframe-safe
# scalar value.
# ------------------------------------------------------------

def make_cell_safe(value: Any) -> Any:

    # Missing values
    if value is None:
        return ""

    # Lists / tuples / arrays
    if isinstance(
        value,
        (list, tuple, np.ndarray)
    ):

        cleaned_items = []

        for item in value:

            if item is None:
                continue

            if isinstance(
                item,
                (float, np.floating)
            ) and pd.isna(item):

                continue

            cleaned_items.append(
                str(item).strip()
            )

        return " | ".join(
            item
            for item in cleaned_items
            if item
        )

    # Dictionaries
    if isinstance(
        value,
        dict
    ):

        try:

            return json.dumps(
                value,
                ensure_ascii=False
            )

        except Exception:

            return str(value)

    # NumPy scalar
    if isinstance(
        value,
        np.generic
    ):

        return value.item()

    # Pandas missing value
    try:

        if pd.isna(value):

            return ""

    except (TypeError, ValueError):

        pass

    return value


# ------------------------------------------------------------
# Add WebShield output columns to the integrated dataset
# ------------------------------------------------------------

for column in WEB_OUTPUT_FIELDS:

    target_column = f"web_{column}"

    if target_column not in integrated_supply.columns:

        if column in [
            "action"
        ]:

            integrated_supply[
                target_column
            ] = "NORMAL"

        elif column in [
            "webshield_reasons"
        ]:

            integrated_supply[
                target_column
            ] = ""

        else:

            integrated_supply[
                target_column
            ] = 0.0


# ------------------------------------------------------------
# Transfer matched WebShield records
# ------------------------------------------------------------

matched_count = 0
unmatched_count = 0


for supply_index, webshield_index in (
    integrated_supply[
        "webshield_match_index"
    ].items()
):

    # --------------------------------------------------------
    # Handle unmatched records
    # --------------------------------------------------------

    if pd.isna(webshield_index):

        unmatched_count += 1

        continue


    # --------------------------------------------------------
    # Convert index safely
    # --------------------------------------------------------

    try:

        webshield_index = int(
            webshield_index
        )

    except (
        TypeError,
        ValueError
    ):

        unmatched_count += 1

        continue


    # --------------------------------------------------------
    # Make sure referenced WebShield row exists
    # --------------------------------------------------------

    if webshield_index not in integrated_webshield.index:

        unmatched_count += 1

        continue


    web_row = integrated_webshield.loc[
        webshield_index
    ]


    # --------------------------------------------------------
    # Transfer every WebShield signal
    # --------------------------------------------------------

    for column in WEB_OUTPUT_FIELDS:

        target_column = f"web_{column}"

        raw_value = web_row.get(
            column,
            ""
        )


        # Convert lists/dicts/arrays to ONE scalar string.
        safe_value = make_cell_safe(
            raw_value
        )


        # ----------------------------------------------------
        # IMPORTANT:
        # .at[] forces a single-cell assignment.
        # This prevents the Pandas iterable-assignment error.
        # ----------------------------------------------------

        integrated_supply.at[
            supply_index,
            target_column
        ] = safe_value


    matched_count += 1


# ------------------------------------------------------------
# Normalize WebShield numerical signals
# ------------------------------------------------------------

for column in [
    "web_webshield_risk",
    "web_counterfeit_risk"
]:

    integrated_supply[
        column
    ] = (
        pd.to_numeric(
            integrated_supply[
                column
            ],
            errors="coerce"
        )
        .fillna(0)
        .clip(
            0,
            100
        )
    )


# ------------------------------------------------------------
# Normalize component signals
# ------------------------------------------------------------

for column in [
    "web_price_anomaly",
    "web_seller_risk",
    "web_review_anomaly",
    "web_similarity_risk",
    "web_listing_quality"
]:

    integrated_supply[
        column
    ] = (
        pd.to_numeric(
            integrated_supply[
                column
            ],
            errors="coerce"
        )
        .fillna(0)
        .clip(
            0,
            1
        )
    )


# ------------------------------------------------------------
# Normalize action field
# ------------------------------------------------------------

integrated_supply[
    "web_action"
] = (
    integrated_supply[
        "web_action"
    ]
    .apply(
        make_cell_safe
    )
    .replace(
        "",
        "NORMAL"
    )
)


# ------------------------------------------------------------
# Normalize WebShield reasons
# ------------------------------------------------------------

integrated_supply[
    "web_webshield_reasons"
] = (
    integrated_supply[
        "web_webshield_reasons"
    ]
    .apply(
        make_cell_safe
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

required_columns = [
    "web_webshield_risk",
    "web_counterfeit_risk",
    "web_price_anomaly",
    "web_seller_risk",
    "web_review_anomaly",
    "web_similarity_risk",
    "web_listing_quality",
    "web_action",
    "web_webshield_reasons"
]


missing_columns = [
    column
    for column in required_columns
    if column not in integrated_supply.columns
]


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("=" * 90)
print("WEBSHIELD INTELLIGENCE MERGE")
print("=" * 90)

print(
    f"Supply-chain records       : "
    f"{len(integrated_supply):,}"
)

print(
    f"WebShield records          : "
    f"{len(integrated_webshield):,}"
)

print(
    f"Matched records            : "
    f"{matched_count:,}"
)

print(
    f"Unmatched records          : "
    f"{unmatched_count:,}"
)

print(
    f"Required columns available : "
    f"{len(required_columns) - len(missing_columns)}"
    f"/{len(required_columns)}"
)


if missing_columns:

    print()
    print(
        "WARNING — Missing columns:"
    )

    for column in missing_columns:

        print(
            f"  - {column}"
        )

else:

    print()
    print(
        "All WebShield output columns are present."
    )


# ------------------------------------------------------------
# Show sample
# ------------------------------------------------------------

print()
print("=" * 90)
print("WEBSHIELD MERGE SAMPLE")
print("=" * 90)


display(
    integrated_supply[
        [
            "supplier",
            "product",
            "web_webshield_risk",
            "web_counterfeit_risk",
            "web_price_anomaly",
            "web_seller_risk",
            "web_review_anomaly",
            "web_action",
            "web_webshield_reasons",
            "webshield_match_method"
        ]
    ]
    .head(10)
)


print()
print("=" * 90)
print("CELL 16 COMPLETED SUCCESSFULLY")
print("=" * 90)

WEBSHIELD INTELLIGENCE MERGE
Supply-chain records       : 303
WebShield records          : 303
Matched records            : 303
Unmatched records          : 0
Required columns available : 9/9

All WebShield output columns are present.

WEBSHIELD MERGE SAMPLE


,supplier,product,web_webshield_risk,web_counterfeit_risk,web_price_anomaly,web_seller_risk,web_review_anomaly,web_action,web_webshield_reasons,webshield_match_method
0,,Wall Mount Mop Holder – No-Slide Grip for Home...,15.66,15.27,0.061392,0.3280,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
1,,Compact TianMu Tool Set – Essential 9-Piece Re...,16.76,16.70,0.079600,0.3140,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
2,,Manual Wall Fastening Nail Gun Tool Set (1 Set),25.63,27.27,0.198896,0.3217,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
3,,Mini Precision Screwdriver Set – Compact & Mul...,20.05,20.75,0.106653,0.3364,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
4,,Electric Drill Machine – Compact & Powerful 28...,19.75,20.00,0.129303,0.3203,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
5,,Leak Proof Tape – Instant Waterproof Seal for ...,18.75,18.14,0.113467,0.3777,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
6,,Metal Try Square Ruler Set – Durable 2-Piece P...,16.54,16.12,0.087140,0.3203,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
7,,Heavy Duty Sledge Hammer Rubber Mallet for Con...,17.81,17.75,0.100701,0.3189,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
8,,Hardware Tool Set – 11 Pcs Multi-Functional Kit,18.87,18.80,0.104985,0.3539,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT
9,,Heavy Duty Electric Drill – Powerful & Versati...,20.15,20.40,0.129303,0.3364,0.0,NORMAL,No strong WebShield anomaly detected,COMPANY_PRODUCT



CELL 16 COMPLETED SUCCESSFULLY


## 5. Unified SupplyShield Risk

The final application risk combines the supply-chain intelligence layer with
the WebShield layer.

The integrated score remains explainable:

- Supply-chain risk: 60%
- WebShield risk: 25%
- Counterfeit-risk proxy: 15%

These weights represent an application-level decision policy rather than a
claim that these proportions are universally optimal.

In [16]:
# ============================================================
# CELL — ROBUST CLIP SCORE UTILITY
# ============================================================

def clip_score(value, fallback=0.0):
    """
    Safely converts a value into a normalized CLIP-style risk score.

    Parameters
    ----------
    value : Any
        Input value. Can be numeric, string, None, NaN,
        list, numpy array, etc.

    fallback : float, optional
        Fallback score used when the input cannot be interpreted.

    Returns
    -------
    float
        Score normalized to the range [0, 100].
    """

    # --------------------------------------------------------
    # Handle None
    # --------------------------------------------------------

    if value is None:
        return float(fallback)

    # --------------------------------------------------------
    # Handle pandas missing values safely
    # --------------------------------------------------------

    try:
        if pd.isna(value):
            return float(fallback)
    except (TypeError, ValueError):
        pass

    # --------------------------------------------------------
    # Handle numpy arrays / lists / tuples
    # --------------------------------------------------------

    if isinstance(value, (list, tuple, np.ndarray)):

        try:

            arr = np.asarray(value)

            # Empty array
            if arr.size == 0:
                return float(fallback)

            # Flatten
            arr = arr.reshape(-1)

            # Keep only numeric values
            numeric_values = pd.to_numeric(
                pd.Series(arr),
                errors="coerce"
            ).dropna()

            if numeric_values.empty:
                return float(fallback)

            # Mean if multiple values exist
            numeric_value = float(
                numeric_values.mean()
            )

        except Exception:
            return float(fallback)

    # --------------------------------------------------------
    # Handle scalar numeric values
    # --------------------------------------------------------

    else:

        try:
            numeric_value = float(value)

        except (TypeError, ValueError):
            return float(fallback)

    # --------------------------------------------------------
    # Protect against NaN / infinity
    # --------------------------------------------------------

    if not np.isfinite(numeric_value):
        return float(fallback)

    # --------------------------------------------------------
    # Normalize score
    #
    # Supports:
    #   0–1   → converted to 0–100
    #   0–100 → kept as 0–100
    # --------------------------------------------------------

    if 0.0 <= numeric_value <= 1.0:

        numeric_value *= 100.0

    # --------------------------------------------------------
    # Final safety clipping
    # --------------------------------------------------------

    numeric_value = np.clip(
        numeric_value,
        0.0,
        100.0
    )

    return float(numeric_value)


# ------------------------------------------------------------
# FUNCTION VALIDATION
# ------------------------------------------------------------

_test_values = [
    None,
    np.nan,
    0,
    0.25,
    0.75,
    1.0,
    25,
    75,
    100,
    [],
    np.array([]),
    np.array([0.25, 0.50, 0.75]),
]

print("=" * 75)
print("CLIP SCORE UTILITY VALIDATION")
print("=" * 75)

for value in _test_values:

    result = clip_score(value)

    print(
        f"Input: {str(value):<35} "
        f"→ Score: {result:.2f}"
    )

print()
print("Two-argument compatibility test:")

print(
    "clip_score(0.75, 0.0) →",
    clip_score(0.75, 0.0)
)

print(
    "clip_score([], 0.0)   →",
    clip_score([], 0.0)
)

print()
print("✓ CLIP SCORE UTILITY VALIDATED")

CLIP SCORE UTILITY VALIDATION
Input: None                                → Score: 0.00
Input: nan                                 → Score: 0.00
Input: 0                                   → Score: 0.00
Input: 0.25                                → Score: 25.00
Input: 0.75                                → Score: 75.00
Input: 1.0                                 → Score: 100.00
Input: 25                                  → Score: 25.00
Input: 75                                  → Score: 75.00
Input: 100                                 → Score: 100.00
Input: []                                  → Score: 0.00
Input: []                                  → Score: 0.00
Input: [0.25 0.5  0.75]                    → Score: 50.00

Two-argument compatibility test:
clip_score(0.75, 0.0) → 75.0
clip_score([], 0.0)   → 0.0

✓ CLIP SCORE UTILITY VALIDATED


In [18]:
# ============================================================
# CELL 19 — UNIFIED RISK VALIDATION & EXECUTIVE SUMMARY
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Validate upstream dataframe
# ------------------------------------------------------------

if "integrated_supply" not in globals():

    raise RuntimeError(
        "integrated_supply dataframe is not available. "
        "Please execute the previous integration cells first."
    )


if integrated_supply.empty:

    raise ValueError(
        "integrated_supply is empty. "
        "No records are available for risk analysis."
    )


# ------------------------------------------------------------
# 2. Required score column
# ------------------------------------------------------------

RISK_SCORE_COLUMN = "unified_supplyshield_risk"


if RISK_SCORE_COLUMN not in integrated_supply.columns:

    raise KeyError(
        f"Required column '{RISK_SCORE_COLUMN}' was not found. "
        "Please execute Cell 18 successfully before Cell 19."
    )


# ------------------------------------------------------------
# 3. Defensive numeric conversion
# ------------------------------------------------------------

integrated_supply[
    RISK_SCORE_COLUMN
] = pd.to_numeric(
    integrated_supply[
        RISK_SCORE_COLUMN
    ],
    errors="coerce"
).fillna(0.0)


# ------------------------------------------------------------
# 4. Enforce valid risk range
# ------------------------------------------------------------

integrated_supply[
    RISK_SCORE_COLUMN
] = (
    integrated_supply[
        RISK_SCORE_COLUMN
    ]
    .clip(0, 100)
    .round(2)
)


# ------------------------------------------------------------
# 5. Robust risk-band generator
# ------------------------------------------------------------

def generate_risk_band(score):
    """
    Convert a 0–100 risk score into an interpretable
    enterprise risk band.
    """

    try:

        score = float(score)

    except (TypeError, ValueError):

        return "UNKNOWN"


    if not np.isfinite(score):

        return "UNKNOWN"


    if score >= 80:

        return "CRITICAL"

    elif score >= 60:

        return "HIGH"

    elif score >= 40:

        return "MEDIUM"

    elif score >= 20:

        return "LOW"

    else:

        return "MINIMAL"


# ------------------------------------------------------------
# 6. ALWAYS regenerate the risk band
#
# This deliberately does not depend on an older notebook
# version of unified_risk_band.
# ------------------------------------------------------------

integrated_supply[
    "unified_risk_band"
] = (
    integrated_supply[
        RISK_SCORE_COLUMN
    ]
    .apply(generate_risk_band)
)


# ------------------------------------------------------------
# 7. Generate risk priority
# ------------------------------------------------------------

risk_priority_mapping = {

    "CRITICAL": 5,
    "HIGH": 4,
    "MEDIUM": 3,
    "LOW": 2,
    "MINIMAL": 1,
    "UNKNOWN": 0

}


integrated_supply[
    "risk_priority"
] = (
    integrated_supply[
        "unified_risk_band"
    ]
    .map(risk_priority_mapping)
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 8. Validation checks
# ------------------------------------------------------------

assert (
    integrated_supply[
        RISK_SCORE_COLUMN
    ]
    .between(0, 100)
    .all()
), "Risk score validation failed."


assert (
    integrated_supply[
        "unified_risk_band"
    ]
    .notna()
    .all()
), "Risk-band generation failed."


# ------------------------------------------------------------
# 9. Overall portfolio metrics
# ------------------------------------------------------------

total_records = len(integrated_supply)

average_risk = (
    integrated_supply[
        RISK_SCORE_COLUMN
    ]
    .mean()
)


median_risk = (
    integrated_supply[
        RISK_SCORE_COLUMN
    ]
    .median()
)


maximum_risk = (
    integrated_supply[
        RISK_SCORE_COLUMN
    ]
    .max()
)


minimum_risk = (
    integrated_supply[
        RISK_SCORE_COLUMN
    ]
    .min()
)


critical_count = int(
    (
        integrated_supply[
            "unified_risk_band"
        ] == "CRITICAL"
    ).sum()
)


high_count = int(
    (
        integrated_supply[
            "unified_risk_band"
        ] == "HIGH"
    ).sum()
)


medium_count = int(
    (
        integrated_supply[
            "unified_risk_band"
        ] == "MEDIUM"
    ).sum()
)


low_count = int(
    (
        integrated_supply[
            "unified_risk_band"
        ] == "LOW"
    ).sum()
)


minimal_count = int(
    (
        integrated_supply[
            "unified_risk_band"
        ] == "MINIMAL"
    ).sum()
)


# ------------------------------------------------------------
# 10. Executive summary
# ------------------------------------------------------------

print()
print("=" * 90)
print("SUPPLYSHIELD AI — UNIFIED RISK VALIDATION")
print("=" * 90)

print()

print(
    f"Total records              : {total_records:,}"
)

print(
    f"Average risk score         : {average_risk:.2f}/100"
)

print(
    f"Median risk score          : {median_risk:.2f}/100"
)

print(
    f"Maximum risk score        : {maximum_risk:.2f}/100"
)

print(
    f"Minimum risk score        : {minimum_risk:.2f}/100"
)

print()

print("RISK DISTRIBUTION")
print("-" * 90)

print(
    f"CRITICAL                  : {critical_count:,}"
)

print(
    f"HIGH                      : {high_count:,}"
)

print(
    f"MEDIUM                    : {medium_count:,}"
)

print(
    f"LOW                       : {low_count:,}"
)

print(
    f"MINIMAL                   : {minimal_count:,}"
)


# ------------------------------------------------------------
# 11. Risk distribution dataframe
# ------------------------------------------------------------

risk_distribution = (
    integrated_supply[
        "unified_risk_band"
    ]
    .value_counts()
    .rename_axis("risk_band")
    .reset_index(
        name="record_count"
    )
)


risk_distribution[
    "percentage"
] = (
    risk_distribution[
        "record_count"
    ]
    / total_records
    * 100
).round(2)


print()
print("RISK DISTRIBUTION TABLE")
print("-" * 90)

display(
    risk_distribution
)


# ------------------------------------------------------------
# 12. Identify highest-risk records
# ------------------------------------------------------------

display_columns = [

    "supplier",
    "company",
    "product",

    "component_supply_chain_risk",
    "component_webshield_risk",
    "component_counterfeit_proxy",

    "unified_supplyshield_risk",
    "unified_risk_band",
    "risk_priority"
]


# Keep only columns that actually exist.
available_display_columns = [

    column

    for column in display_columns

    if column in integrated_supply.columns

]


top_risk_records = (
    integrated_supply[
        available_display_columns
    ]
    .sort_values(
        by=[
            "risk_priority",
            RISK_SCORE_COLUMN
        ],
        ascending=[
            False,
            False
        ]
    )
    .head(20)
    .reset_index(drop=True)
)


print()
print("=" * 90)
print("TOP 20 SUPPLYSHIELD RISK RECORDS")
print("=" * 90)

display(
    top_risk_records
)


# ------------------------------------------------------------
# 13. Supplier-level aggregation
# ------------------------------------------------------------

if "supplier" in integrated_supply.columns:

    supplier_risk_summary = (

        integrated_supply

        .groupby(
            "supplier",
            dropna=False
        )

        .agg(

            records=(
                RISK_SCORE_COLUMN,
                "count"
            ),

            average_risk=(
                RISK_SCORE_COLUMN,
                "mean"
            ),

            maximum_risk=(
                RISK_SCORE_COLUMN,
                "max"
            )

        )

        .reset_index()

    )


    supplier_risk_summary[
        "average_risk"
    ] = (
        supplier_risk_summary[
            "average_risk"
        ]
        .round(2)
    )


    supplier_risk_summary[
        "maximum_risk"
    ] = (
        supplier_risk_summary[
            "maximum_risk"
        ]
        .round(2)
    )


    supplier_risk_summary[
        "risk_band"
    ] = (
        supplier_risk_summary[
            "average_risk"
        ]
        .apply(generate_risk_band)
    )


    supplier_risk_summary = (

        supplier_risk_summary

        .sort_values(
            "average_risk",
            ascending=False
        )

        .reset_index(drop=True)

    )


    print()
    print("=" * 90)
    print("SUPPLIER-LEVEL RISK SUMMARY")
    print("=" * 90)

    display(
        supplier_risk_summary.head(20)
    )


# ------------------------------------------------------------
# 14. Final validation report
# ------------------------------------------------------------

print()
print("=" * 90)
print("CELL 19 VALIDATION")
print("=" * 90)

print(
    f"✓ Risk score column available     : "
    f"{RISK_SCORE_COLUMN}"
)

print(
    f"✓ Risk scores normalized          : 0–100"
)

print(
    f"✓ Risk bands generated            : "
    f"{integrated_supply['unified_risk_band'].nunique()}"
)

print(
    f"✓ Risk priority generated         : "
    f"{'risk_priority' in integrated_supply.columns}"
)

print(
    f"✓ Records validated               : "
    f"{len(integrated_supply):,}"
)

print()
print("✓ CELL 19 COMPLETED SUCCESSFULLY")
print("=" * 90)


SUPPLYSHIELD AI — UNIFIED RISK VALIDATION

Total records              : 303
Average risk score         : 10.95/100
Median risk score          : 9.45/100
Maximum risk score        : 23.38/100
Minimum risk score        : 6.30/100

RISK DISTRIBUTION
------------------------------------------------------------------------------------------
CRITICAL                  : 0
HIGH                      : 0
MEDIUM                    : 0
LOW                       : 6
MINIMAL                   : 297

RISK DISTRIBUTION TABLE
------------------------------------------------------------------------------------------


,risk_band,record_count,percentage
0,MINIMAL,297,98.02
1,LOW,6,1.98



TOP 20 SUPPLYSHIELD RISK RECORDS


,supplier,company,product,component_supply_chain_risk,component_webshield_risk,component_counterfeit_proxy,unified_supplyshield_risk,unified_risk_band,risk_priority
0,,Arihant Medmach Private Limited,Siemens Artis Zee Cath Lab,11.13,41.63,41.96,23.38,LOW,2
1,,Rentomed,Allenger Altima F100 Fixed Cath Lab Machine,11.00,38.34,37.03,21.74,LOW,2
2,,Mf India,Cath Lab Machine,10.67,38.34,37.03,21.54,LOW,2
3,,Applied Techno Engineers Private Limited,Broken Bag Detector,9.62,39.57,38.87,21.50,LOW,2
4,,Soarmlich Engineers,10mm Broken Bag Detector,9.08,39.12,38.19,20.96,LOW,2
5,,Hnl Systems Pvt. Ltd.,Broken Bag Detector,9.49,38.41,37.13,20.87,LOW,2
6,,Kabir Industrial Technology,Front Belt 10 Bricks Making Machine,7.95,35.94,33.43,18.77,MINIMAL,1
7,,Vpg Buildwell India Pvt. Ltd.,Mini Dumper,7.69,35.08,32.13,18.20,MINIMAL,1
8,,App Pumps Engineering Company,Garco Back Pressure Regulator,3.89,39.60,38.92,18.07,MINIMAL,1
9,,Parul Engineering Private Limited,Air Slide,5.07,37.98,36.49,18.01,MINIMAL,1



SUPPLIER-LEVEL RISK SUMMARY


,supplier,records,average_risk,maximum_risk,risk_band
0,,303,10.95,23.38,MINIMAL



CELL 19 VALIDATION
✓ Risk score column available     : unified_supplyshield_risk
✓ Risk scores normalized          : 0–100
✓ Risk bands generated            : 2
✓ Risk priority generated         : True
✓ Records validated               : 303

✓ CELL 19 COMPLETED SUCCESSFULLY


In [19]:
# ============================================================
# CELL 20 — OPERATIONAL PRIORITY ENGINE
# ============================================================

def calculate_priority(
    row: pd.Series
) -> float:

    unified = safe_float(
        row.get(
            "unified_supplyshield_risk"
        ),
        0
    )

    web = safe_float(
        row.get(
            "component_webshield_risk"
        ),
        0
    )

    counterfeit = safe_float(
        row.get(
            "component_counterfeit_proxy"
        ),
        0
    )

    disruption = safe_float(
        row.get(
            "disruption_risk"
        ),
        0
    )

    # Priority emphasizes immediate operational threats.
    priority = (
        0.45 * unified
        + 0.20 * web
        + 0.20 * counterfeit
        + 0.15 * disruption
    )

    return float(
        np.clip(
            priority,
            0,
            100
        )
    )


integrated_supply[
    "operational_priority_score"
] = (
    integrated_supply
    .apply(
        calculate_priority,
        axis=1
    )
    .round(2)
)


integrated_supply[
    "operational_priority_band"
] = (
    integrated_supply[
        "operational_priority_score"
    ]
    .apply(score_to_band)
)


priority_view = (
    integrated_supply[
        [
            "supplier",
            "product",
            "unified_supplyshield_risk",
            "unified_risk_band",
            "component_webshield_risk",
            "component_counterfeit_proxy",
            "operational_priority_score",
            "operational_priority_band"
        ]
    ]
    .sort_values(
        "operational_priority_score",
        ascending=False
    )
    .reset_index(drop=True)
)


priority_view.insert(
    0,
    "priority_rank",
    np.arange(
        1,
        len(priority_view) + 1
    )
)


print("=" * 90)
print("TOP OPERATIONAL PRIORITIES")
print("=" * 90)

display(
    priority_view.head(20)
)

TOP OPERATIONAL PRIORITIES


,priority_rank,supplier,product,unified_supplyshield_risk,unified_risk_band,component_webshield_risk,component_counterfeit_proxy,operational_priority_score,operational_priority_band
0,1,,Siemens Artis Zee Cath Lab,23.38,LOW,41.63,41.96,27.24,LOW
1,2,,Broken Bag Detector,21.50,LOW,39.57,38.87,25.36,LOW
2,3,,10mm Broken Bag Detector,20.96,LOW,39.12,38.19,24.89,LOW
3,4,,Allenger Altima F100 Fixed Cath Lab Machine,21.74,LOW,38.34,37.03,24.86,LOW
4,5,,Cath Lab Machine,21.54,LOW,38.34,37.03,24.77,LOW
5,6,,Broken Bag Detector,20.87,LOW,38.41,37.13,24.50,LOW
6,7,,Hydraulic Pallet Truck,17.89,MINIMAL,40.05,39.60,23.98,LOW
7,8,,Hydraulic Pallet Truck,17.72,MINIMAL,40.05,39.60,23.90,LOW
8,9,,Garco Back Pressure Regulator,18.07,MINIMAL,39.60,38.92,23.84,LOW
9,10,,Silver Chloride,17.63,MINIMAL,39.95,38.46,23.62,LOW


## 6. Dashboard-Ready Unified Dataset

The following dataset is the primary output that the future Streamlit
application can consume.

It contains both component intelligence and the final integrated risk,
allowing the UI to explain *why* a supplier/product received its score.

In [20]:
# ============================================================
# CELL 22 — BUILD DASHBOARD-READY DATASET
# ============================================================

DASHBOARD_COLUMNS = [

    # Identity
    "supplier",
    "company",
    "product",
    "title",
    "source",
    "location",

    # Supply-chain intelligence
    "supply_chain_risk",
    "supply_chain_risk_band",
    "nlp_risk",
    "supplier_risk",
    "anomaly_risk",
    "disruption_risk",
    "market_risk",

    # WebShield intelligence
    "web_webshield_risk",
    "web_counterfeit_risk",
    "web_price_anomaly",
    "web_seller_risk",
    "web_review_anomaly",
    "web_similarity_risk",
    "web_listing_quality",
    "web_action",

    # Integrated intelligence
    "component_supply_chain_risk",
    "component_webshield_risk",
    "component_counterfeit_proxy",

    "unified_supplyshield_risk",
    "unified_risk_band",

    "operational_priority_score",
    "operational_priority_band",

    # Explainability
    "supply_chain_reasons",
    "web_webshield_reasons",
    "unified_risk_reasons",

    # Matching
    "webshield_match_method"
]


available_dashboard_columns = [
    column
    for column in DASHBOARD_COLUMNS
    if column in integrated_supply.columns
]


dashboard_df = (
    integrated_supply[
        available_dashboard_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# Convert list explanations to readable strings
# ------------------------------------------------------------

for column in [
    "unified_risk_reasons"
]:

    if column in dashboard_df.columns:

        dashboard_df[column] = (
            dashboard_df[column]
            .apply(
                lambda value:
                " | ".join(value)
                if isinstance(value, list)
                else safe_text(value)
            )
        )


print("=" * 85)
print("DASHBOARD DATASET CREATED")
print("=" * 85)

print(
    f"Rows    : {len(dashboard_df):,}"
)

print(
    f"Columns : {len(dashboard_df.columns):,}"
)

display(
    dashboard_df.head(10)
)

DASHBOARD DATASET CREATED
Rows    : 303
Columns : 32


,supplier,company,product,title,source,location,supply_chain_risk,supply_chain_risk_band,nlp_risk,supplier_risk,...,component_webshield_risk,component_counterfeit_proxy,unified_supplyshield_risk,unified_risk_band,operational_priority_score,operational_priority_band,supply_chain_reasons,web_webshield_reasons,unified_risk_reasons,webshield_match_method
0,,GlimmerHome,Wall Mount Mop Holder – No-Slide Grip for Home...,Wall Mount Mop Holder – No-Slide Grip for Home...,DeoDap,,8.28,LOW,0.050000,0.0,...,15.66,15.27,11.17,MINIMAL,11.21,MINIMAL,Unusual/anomalous behaviour detected (72.8/100),No strong WebShield anomaly detected,Significant anomaly signal detected,COMPANY_PRODUCT
1,,DeoDap,Compact TianMu Tool Set – Essential 9-Piece Re...,Compact TianMu Tool Set – Essential 9-Piece Re...,DeoDap,,8.37,LOW,0.050000,0.0,...,16.76,16.70,11.72,MINIMAL,11.97,MINIMAL,Unusual/anomalous behaviour detected (73.7/100),No strong WebShield anomaly detected,Significant anomaly signal detected,COMPANY_PRODUCT
2,,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),Manual Wall Fastening Nail Gun Tool Set (1 Set),DeoDap,,9.66,LOW,0.079295,0.0,...,25.63,27.27,16.29,MINIMAL,17.91,MINIMAL,Unusual/anomalous behaviour detected (80.7/100),No strong WebShield anomaly detected,Significant anomaly signal detected,COMPANY_PRODUCT
3,,DeoDap,Mini Precision Screwdriver Set – Compact & Mul...,Mini Precision Screwdriver Set – Compact & Mul...,DeoDap,,8.04,LOW,0.042345,0.0,...,20.05,20.75,12.95,MINIMAL,13.99,MINIMAL,Unusual/anomalous behaviour detected (72.0/100),No strong WebShield anomaly detected,Significant anomaly signal detected,COMPANY_PRODUCT
4,,DeoDap,Electric Drill Machine – Compact & Powerful 28...,Electric Drill Machine – Compact & Powerful 28...,DeoDap,,8.82,LOW,0.015960,0.0,...,19.75,20.00,13.23,MINIMAL,13.90,MINIMAL,Unusual/anomalous behaviour detected (85.0/100),No strong WebShield anomaly detected,Significant anomaly signal detected,COMPANY_PRODUCT
5,,BoltForce,Leak Proof Tape – Instant Waterproof Seal for ...,Leak Proof Tape – Instant Waterproof Seal for ...,DeoDap,,11.45,LOW,0.079295,0.0,...,18.75,18.14,14.28,MINIMAL,13.80,MINIMAL,Unusual/anomalous behaviour detected (98.7/100),No strong WebShield anomaly detected,Significant anomaly signal detected,COMPANY_PRODUCT
6,,DeoDap,Metal Try Square Ruler Set – Durable 2-Piece P...,Metal Try Square Ruler Set – Durable 2-Piece P...,DeoDap,,7.61,LOW,0.050000,0.0,...,16.54,16.12,11.12,MINIMAL,11.54,MINIMAL,Unusual/anomalous behaviour detected (66.1/100),No strong WebShield anomaly detected,No strong integrated risk driver detected,COMPANY_PRODUCT
7,,DeoDap,Heavy Duty Sledge Hammer Rubber Mallet for Con...,Heavy Duty Sledge Hammer Rubber Mallet for Con...,DeoDap,,7.95,LOW,0.050000,0.0,...,17.81,17.75,11.88,MINIMAL,12.46,MINIMAL,Unusual/anomalous behaviour detected (69.5/100),No strong WebShield anomaly detected,No strong integrated risk driver detected,COMPANY_PRODUCT
8,,BoltForce,Hardware Tool Set – 11 Pcs Multi-Functional Kit,Hardware Tool Set – 11 Pcs Multi-Functional Ki...,DeoDap,,9.48,LOW,0.050000,0.0,...,18.87,18.80,13.23,MINIMAL,13.49,MINIMAL,Unusual/anomalous behaviour detected (84.8/100),No strong WebShield anomaly detected,Significant anomaly signal detected,COMPANY_PRODUCT
9,,DeoDap,Heavy Duty Electric Drill – Powerful & Versati...,Heavy Duty Electric Drill – Powerful & Versati...,DeoDap,,7.58,LOW,0.015960,0.0,...,20.15,20.40,12.65,MINIMAL,13.80,MINIMAL,Unusual/anomalous behaviour detected (72.6/100),No strong WebShield anomaly detected,Significant anomaly signal detected,COMPANY_PRODUCT


In [21]:
# ============================================================
# CELL 23 — EXECUTIVE DASHBOARD METRICS
# ============================================================

total_records = len(
    dashboard_df
)

high_or_critical = (
    dashboard_df[
        "unified_risk_band"
    ]
    .isin(
        [
            "HIGH",
            "CRITICAL"
        ]
    )
    .sum()
)


webshield_alerts = (
    dashboard_df[
        "web_webshield_risk"
    ]
    >= 60
).sum()


counterfeit_alerts = (
    dashboard_df[
        "web_counterfeit_risk"
    ]
    >= 60
).sum()


critical_supply = (
    dashboard_df[
        "unified_risk_band"
    ]
    == "CRITICAL"
).sum()


average_risk = (
    dashboard_df[
        "unified_supplyshield_risk"
    ]
    .mean()
)


maximum_risk = (
    dashboard_df[
        "unified_supplyshield_risk"
    ]
    .max()
)


executive_metrics = pd.DataFrame({

    "metric": [
        "Total intelligence records",
        "Average unified risk",
        "Maximum unified risk",
        "High/Critical records",
        "Critical records",
        "WebShield alerts",
        "Counterfeit-risk alerts"
    ],

    "value": [
        int(total_records),
        round(float(average_risk), 2),
        round(float(maximum_risk), 2),
        int(high_or_critical),
        int(critical_supply),
        int(webshield_alerts),
        int(counterfeit_alerts)
    ]
})


display(
    executive_metrics
)

,metric,value
0,Total intelligence records,303.00
1,Average unified risk,10.95
2,Maximum unified risk,23.38
3,High/Critical records,0.00
4,Critical records,0.00
5,WebShield alerts,0.00
6,Counterfeit-risk alerts,0.00


In [22]:
# ============================================================
# CELL 24 — UNIFIED RISK DISTRIBUTION
# ============================================================

risk_distribution = (
    dashboard_df[
        "unified_risk_band"
    ]
    .value_counts()
    .reindex(
        [
            "CRITICAL",
            "HIGH",
            "MEDIUM",
            "LOW",
            "MINIMAL"
        ],
        fill_value=0
    )
    .rename_axis(
        "risk_band"
    )
    .reset_index(
        name="record_count"
    )
)


risk_distribution[
    "percentage"
] = (
    risk_distribution[
        "record_count"
    ]
    / max(
        len(dashboard_df),
        1
    )
    * 100
).round(2)


display(
    risk_distribution
)

,risk_band,record_count,percentage
0,CRITICAL,0,0.00
1,HIGH,0,0.00
2,MEDIUM,0,0.00
3,LOW,6,1.98
4,MINIMAL,297,98.02


## 7. SQL Integration

Notebook 06 established the relational database.

This stage does not rebuild the database. It adds the integrated
SupplyShield intelligence so that the future application can query one
persistent source of truth.

In [23]:
# ============================================================
# CELL 26 — CONNECT TO EXISTING SQL DATABASE
# ============================================================

if not DATABASE_PATH.exists():

    raise FileNotFoundError(
        f"Database not found:\n{DATABASE_PATH}\n\n"
        "Run Notebook 06 first."
    )


connection = sqlite3.connect(
    DATABASE_PATH
)

connection.execute(
    "PRAGMA foreign_keys = ON;"
)

print("=" * 85)
print("SQL DATABASE CONNECTED")
print("=" * 85)

print(
    f"Database: {DATABASE_PATH}"
)

print(
    f"SQLite version: {sqlite3.sqlite_version}"
)

SQL DATABASE CONNECTED
Database: c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\database\supplyshield.db
SQLite version: 3.45.3


In [24]:
# ============================================================
# CELL 27 — CREATE INTEGRATED RISK TABLE
# ============================================================

cursor = connection.cursor()


cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS integrated_risk (

        integration_id INTEGER PRIMARY KEY AUTOINCREMENT,

        supplier TEXT,

        company TEXT,

        product TEXT,

        title TEXT,

        source TEXT,

        location TEXT,

        supply_chain_risk REAL,

        nlp_risk REAL,

        supplier_risk REAL,

        anomaly_risk REAL,

        disruption_risk REAL,

        market_risk REAL,

        webshield_risk REAL,

        counterfeit_risk REAL,

        price_anomaly REAL,

        seller_risk REAL,

        review_anomaly REAL,

        similarity_risk REAL,

        listing_quality REAL,

        unified_risk REAL,

        unified_risk_band TEXT,

        operational_priority REAL,

        operational_priority_band TEXT,

        webshield_match_method TEXT,

        supply_chain_reasons TEXT,

        webshield_reasons TEXT,

        unified_risk_reasons TEXT,

        integration_timestamp TEXT
    );
    """
)


connection.commit()


print(
    "Integrated risk table created successfully."
)

Integrated risk table created successfully.


In [25]:
# ============================================================
# CELL 28 — INSERT INTEGRATED RISK RECORDS
# ============================================================

insert_columns = [
    "supplier",
    "company",
    "product",
    "title",
    "source",
    "location",

    "supply_chain_risk",
    "nlp_risk",
    "supplier_risk",
    "anomaly_risk",
    "disruption_risk",
    "market_risk",

    "web_webshield_risk",
    "web_counterfeit_risk",
    "web_price_anomaly",
    "web_seller_risk",
    "web_review_anomaly",
    "web_similarity_risk",
    "web_listing_quality",

    "unified_supplyshield_risk",
    "unified_risk_band",

    "operational_priority_score",
    "operational_priority_band",

    "webshield_match_method",

    "supply_chain_reasons",
    "web_webshield_reasons",
    "unified_risk_reasons"
]


sql_insert = """
INSERT INTO integrated_risk (

    supplier,
    company,
    product,
    title,
    source,
    location,

    supply_chain_risk,
    nlp_risk,
    supplier_risk,
    anomaly_risk,
    disruption_risk,
    market_risk,

    webshield_risk,
    counterfeit_risk,
    price_anomaly,
    seller_risk,
    review_anomaly,
    similarity_risk,
    listing_quality,

    unified_risk,
    unified_risk_band,

    operational_priority,
    operational_priority_band,

    webshield_match_method,

    supply_chain_reasons,
    webshield_reasons,
    unified_risk_reasons,

    integration_timestamp

)
VALUES (
    ?, ?, ?, ?, ?, ?,
    ?, ?, ?, ?, ?, ?,
    ?, ?, ?, ?, ?, ?, ?,
    ?, ?,
    ?, ?,
    ?,
    ?, ?, ?,
    ?
)
"""


records_to_insert = []


integration_timestamp = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


for _, row in integrated_supply.iterrows():

    def reasons_to_text(value):

        if isinstance(
            value,
            list
        ):

            return " | ".join(
                str(item)
                for item in value
            )

        return safe_text(value)


    records_to_insert.append(
        (
            safe_text(
                row["supplier"]
            ),

            safe_text(
                row["company"]
            ),

            safe_text(
                row["product"]
            ),

            safe_text(
                row["title"]
            ),

            safe_text(
                row["source"]
            ),

            safe_text(
                row["location"]
            ),

            float(
                row["supply_chain_risk"]
            ),

            float(
                row["nlp_risk"]
            ),

            float(
                row["supplier_risk"]
            ),

            float(
                row["anomaly_risk"]
            ),

            float(
                row["disruption_risk"]
            ),

            float(
                row["market_risk"]
            ),

            float(
                row["web_webshield_risk"]
            ),

            float(
                row["web_counterfeit_risk"]
            ),

            float(
                row["web_price_anomaly"]
            ),

            float(
                row["web_seller_risk"]
            ),

            float(
                row["web_review_anomaly"]
            ),

            float(
                row["web_similarity_risk"]
            ),

            float(
                row["web_listing_quality"]
            ),

            float(
                row["unified_supplyshield_risk"]
            ),

            safe_text(
                row["unified_risk_band"]
            ),

            float(
                row["operational_priority_score"]
            ),

            safe_text(
                row["operational_priority_band"]
            ),

            safe_text(
                row["webshield_match_method"]
            ),

            reasons_to_text(
                row["supply_chain_reasons"]
            ),

            reasons_to_text(
                row["web_webshield_reasons"]
            ),

            reasons_to_text(
                row["unified_risk_reasons"]
            ),

            integration_timestamp
        )
    )


cursor.executemany(
    sql_insert,
    records_to_insert
)

connection.commit()


print("=" * 85)
print("INTEGRATED RECORDS INSERTED")
print("=" * 85)

print(
    f"Records inserted: "
    f"{len(records_to_insert):,}"
)

INTEGRATED RECORDS INSERTED
Records inserted: 303


In [26]:
# ============================================================
# CELL 29 — CREATE DASHBOARD SQL VIEW
# ============================================================

cursor.execute(
    """
    DROP VIEW IF EXISTS vw_supplyshield_dashboard;
    """
)


cursor.execute(
    """
    CREATE VIEW vw_supplyshield_dashboard AS

    SELECT

        integration_id,

        supplier,

        company,

        product,

        title,

        source,

        location,

        supply_chain_risk,

        nlp_risk,

        supplier_risk,

        anomaly_risk,

        disruption_risk,

        market_risk,

        webshield_risk,

        counterfeit_risk,

        price_anomaly,

        seller_risk,

        review_anomaly,

        similarity_risk,

        listing_quality,

        unified_risk,

        unified_risk_band,

        operational_priority,

        operational_priority_band,

        webshield_match_method,

        supply_chain_reasons,

        webshield_reasons,

        unified_risk_reasons,

        integration_timestamp

    FROM integrated_risk;
    """
)


connection.commit()


print(
    "Dashboard SQL view created."
)

Dashboard SQL view created.


In [27]:
# ============================================================
# CELL 30 — VALIDATE SQL INTEGRATION
# ============================================================

sql_dashboard = pd.read_sql_query(
    """
    SELECT *

    FROM vw_supplyshield_dashboard

    ORDER BY
        unified_risk DESC

    LIMIT 20;
    """,
    connection
)


print("=" * 90)
print("TOP INTEGRATED SQL RISK RECORDS")
print("=" * 90)

display(
    sql_dashboard
)

TOP INTEGRATED SQL RISK RECORDS


,integration_id,supplier,company,product,title,source,location,supply_chain_risk,nlp_risk,supplier_risk,...,listing_quality,unified_risk,unified_risk_band,operational_priority,operational_priority_band,webshield_match_method,supply_chain_reasons,webshield_reasons,unified_risk_reasons,integration_timestamp
0,100,,Arihant Medmach Private Limited,Siemens Artis Zee Cath Lab,Siemens Artis Zee Cath Lab - Application: Hosp...,TradeIndia,,11.13,0.061315,0.0,...,0.25,23.38,LOW,27.24,LOW,COMPANY_PRODUCT,Unusual/anomalous behaviour detected (99.0/100),Moderate price anomaly detected | Strong produ...,Significant anomaly signal detected,2026-08-21T05:24:45.403964+00:00
1,128,,Rentomed,Allenger Altima F100 Fixed Cath Lab Machine,Allenger Altima F100 Fixed Cath Lab Machine,TradeIndia,,11.00,0.050000,0.0,...,0.25,21.74,LOW,24.86,LOW,COMPANY_PRODUCT,Unusual/anomalous behaviour detected (100.0/100),Moderate price anomaly detected,Significant anomaly signal detected,2026-08-21T05:24:45.403964+00:00
2,239,,Mf India,Cath Lab Machine,Cath Lab Machine,TradeIndia,,10.67,0.050000,0.0,...,0.25,21.54,LOW,24.77,LOW,COMPANY_PRODUCT,Unusual/anomalous behaviour detected (96.7/100),Moderate price anomaly detected,Significant anomaly signal detected,2026-08-21T05:24:45.403964+00:00
3,43,,Applied Techno Engineers Private Limited,Broken Bag Detector,Broken Bag Detector - Accuracy: +2 %,TradeIndia,,9.62,0.089820,0.0,...,0.25,21.50,LOW,25.36,LOW,COMPANY_PRODUCT,Unusual/anomalous behaviour detected (78.3/100),Moderate price anomaly detected,Significant anomaly signal detected,2026-08-21T05:24:45.403964+00:00
4,136,,Soarmlich Engineers,10mm Broken Bag Detector,10mm Broken Bag Detector - Frequency: 50hz,TradeIndia,,9.08,0.086755,0.0,...,0.25,20.96,LOW,24.89,LOW,COMPANY_PRODUCT,Unusual/anomalous behaviour detected (73.4/100),Moderate price anomaly detected,Significant anomaly signal detected,2026-08-21T05:24:45.403964+00:00
5,201,,Hnl Systems Pvt. Ltd.,Broken Bag Detector,Broken Bag Detector - 315 Grade Stainless Stee...,TradeIndia,,9.49,0.086755,0.0,...,0.25,20.87,LOW,24.50,LOW,COMPANY_PRODUCT,Unusual/anomalous behaviour detected (77.6/100),Moderate price anomaly detected,Significant anomaly signal detected,2026-08-21T05:24:45.403964+00:00
6,219,,Kabir Industrial Technology,Front Belt 10 Bricks Making Machine,Front Belt 10 Bricks Making Machine - Feature:...,TradeIndia,,7.95,0.050000,0.0,...,0.25,18.77,MINIMAL,22.32,LOW,COMPANY_PRODUCT,Unusual/anomalous behaviour detected (69.5/100),Moderate price anomaly detected,No strong integrated risk driver detected,2026-08-21T05:24:45.403964+00:00
7,103,,Vpg Buildwell India Pvt. Ltd.,Mini Dumper,"Mini Dumper - Mild Steel Construction, Yellow ...",TradeIndia,,7.69,0.076335,0.0,...,0.25,18.20,MINIMAL,21.63,LOW,COMPANY_PRODUCT,Unusual/anomalous behaviour detected (61.6/100),Moderate price anomaly detected,No strong integrated risk driver detected,2026-08-21T05:24:45.403964+00:00
8,56,,App Pumps Engineering Company,Garco Back Pressure Regulator,Garco Back Pressure Regulator - Application: I...,TradeIndia,,3.89,0.076335,0.0,...,0.25,18.07,MINIMAL,23.84,LOW,COMPANY_PRODUCT,No individual risk signal exceeded the alert t...,Moderate price anomaly detected,No strong integrated risk driver detected,2026-08-21T05:24:45.403964+00:00
9,74,,Parul Engineering Private Limited,Air Slide,Air Slide - Color: Grey,TradeIndia,,5.07,0.058895,0.0,...,0.25,18.01,MINIMAL,23.00,LOW,COMPANY_PRODUCT,No individual risk signal exceeded the alert t...,Moderate price anomaly detected,No strong integrated risk driver detected,2026-08-21T05:24:45.403964+00:00


## 8. End-to-End Validation

The final integration checkpoint verifies:

- Upstream datasets exist.
- Supply-chain risk is present.
- WebShield risk is present.
- Unified scores remain within 0–100.
- Risk bands are valid.
- SQL records were created.
- Dashboard view is queryable.

In [28]:
# ============================================================
# CELL 32 — END-TO-END VALIDATION
# ============================================================

validation_results = {}


# ------------------------------------------------------------
# Dataset validation
# ------------------------------------------------------------

validation_results[
    "supply_dataset_non_empty"
] = len(supply_df) > 0


validation_results[
    "webshield_dataset_non_empty"
] = len(webshield_df) > 0


validation_results[
    "integration_dataset_non_empty"
] = len(integrated_supply) > 0


# ------------------------------------------------------------
# Risk range validation
# ------------------------------------------------------------

unified_scores = pd.to_numeric(
    integrated_supply[
        "unified_supplyshield_risk"
    ],
    errors="coerce"
)


validation_results[
    "unified_risk_range"
] = bool(
    unified_scores.notna().all()
    and
    (unified_scores >= 0).all()
    and
    (unified_scores <= 100).all()
)


# ------------------------------------------------------------
# WebShield range
# ------------------------------------------------------------

web_scores = pd.to_numeric(
    integrated_supply[
        "web_webshield_risk"
    ],
    errors="coerce"
)


validation_results[
    "webshield_range"
] = bool(
    web_scores.notna().all()
    and
    (web_scores >= 0).all()
    and
    (web_scores <= 100).all()
)


# ------------------------------------------------------------
# Counterfeit proxy range
# ------------------------------------------------------------

counterfeit_scores = pd.to_numeric(
    integrated_supply[
        "web_counterfeit_risk"
    ],
    errors="coerce"
)


validation_results[
    "counterfeit_range"
] = bool(
    counterfeit_scores.notna().all()
    and
    (counterfeit_scores >= 0).all()
    and
    (counterfeit_scores <= 100).all()
)


# ------------------------------------------------------------
# Valid risk bands
# ------------------------------------------------------------

valid_bands = {
    "MINIMAL",
    "LOW",
    "MEDIUM",
    "HIGH",
    "CRITICAL"
}


actual_bands = set(
    integrated_supply[
        "unified_risk_band"
    ]
    .dropna()
    .astype(str)
    .unique()
)


validation_results[
    "risk_bands_valid"
] = (
    actual_bands.issubset(
        valid_bands
    )
)


# ------------------------------------------------------------
# SQL table
# ------------------------------------------------------------

sql_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM integrated_risk;
    """
).fetchone()[0]


validation_results[
    "sql_records_created"
] = (
    sql_count == len(
        integrated_supply
    )
)


# ------------------------------------------------------------
# SQL view
# ------------------------------------------------------------

view_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM vw_supplyshield_dashboard;
    """
).fetchone()[0]


validation_results[
    "dashboard_view_available"
] = (
    view_count == sql_count
)


# ------------------------------------------------------------
# Database integrity
# ------------------------------------------------------------

foreign_key_check = pd.read_sql_query(
    """
    PRAGMA foreign_key_check;
    """,
    connection
)


validation_results[
    "database_integrity"
] = (
    len(foreign_key_check) == 0
)


# ------------------------------------------------------------
# Print report
# ------------------------------------------------------------

print("=" * 90)
print("SUPPLYSHIELD AI — END-TO-END VALIDATION")
print("=" * 90)


all_passed = True


for check, result in validation_results.items():

    status = (
        "PASS"
        if result
        else "FAIL"
    )

    print(
        f"{check:<35} : {status}"
    )

    if not result:
        all_passed = False


print("-" * 90)


if all_passed:

    print(
        "ALL INTEGRATION VALIDATION CHECKS PASSED."
    )

else:

    print(
        "WARNING — ONE OR MORE INTEGRATION CHECKS FAILED."
    )

print("=" * 90)

SUPPLYSHIELD AI — END-TO-END VALIDATION
supply_dataset_non_empty            : PASS
webshield_dataset_non_empty         : PASS
integration_dataset_non_empty       : PASS
unified_risk_range                  : PASS
webshield_range                     : PASS
counterfeit_range                   : PASS
risk_bands_valid                    : PASS
sql_records_created                 : PASS
dashboard_view_available            : PASS
database_integrity                  : PASS
------------------------------------------------------------------------------------------
ALL INTEGRATION VALIDATION CHECKS PASSED.


In [29]:
# ============================================================
# CELL 33 — EXPORT FINAL INTEGRATED DATASET
# ============================================================

FINAL_INTEGRATED_CSV = (
    INTEGRATION_DIR
    / "supplyshield_integrated_risk.csv"
)

FINAL_INTEGRATED_JSON = (
    INTEGRATION_DIR
    / "supplyshield_integrated_risk.json"
)


# ------------------------------------------------------------
# CSV
# ------------------------------------------------------------

dashboard_export = dashboard_df.copy()

dashboard_export.to_csv(
    FINAL_INTEGRATED_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# JSON
# ------------------------------------------------------------

json_records = []

for record in dashboard_df.to_dict(
    orient="records"
):

    safe_record = {}

    for key, value in record.items():

        if isinstance(
            value,
            np.generic
        ):

            value = value.item()

        elif pd.isna(value):

            value = None

        safe_record[key] = value

    json_records.append(
        safe_record
    )


with open(
    FINAL_INTEGRATED_JSON,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        json_records,
        file,
        indent=2,
        ensure_ascii=False
    )


print("=" * 85)
print("FINAL INTEGRATED DATASET EXPORTED")
print("=" * 85)

print(
    f"CSV  : {FINAL_INTEGRATED_CSV}"
)

print(
    f"JSON : {FINAL_INTEGRATED_JSON}"
)

FINAL INTEGRATED DATASET EXPORTED
CSV  : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\integration\supplyshield_integrated_risk.csv
JSON : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\integration\supplyshield_integrated_risk.json


In [30]:
# ============================================================
# CELL 34 — FINAL SYSTEM INTEGRATION REPORT
# ============================================================

database_size_mb = (
    DATABASE_PATH.stat().st_size
    / (1024 ** 2)
)


print()
print("=" * 100)
print("                     SUPPLYSHIELD AI")
print("              SYSTEM INTEGRATION REPORT")
print("=" * 100)

print()
print("INTELLIGENCE PIPELINE")
print("-" * 100)

print(
    "Supply-chain intelligence      : READY"
)

print(
    "NLP / ML risk intelligence    : READY"
)

print(
    "Anomaly intelligence          : READY"
)

print(
    "WebShield intelligence        : READY"
)

print(
    "Unified risk engine           : READY"
)

print(
    "SQL persistence               : READY"
)

print(
    "Dashboard-ready dataset       : READY"
)

print()
print("DATASET")
print("-" * 100)

print(
    f"Integrated records            : "
    f"{len(integrated_supply):,}"
)

print(
    f"WebShield matched records     : "
    f"{(
        integrated_supply[
            'webshield_match_method'
        ] != 'UNMATCHED'
    ).sum():,}"
)

print(
    f"Unmatched records             : "
    f"{(
        integrated_supply[
            'webshield_match_method'
        ] == 'UNMATCHED'
    ).sum():,}"
)

print()
print("RISK")
print("-" * 100)

print(
    f"Average unified risk          : "
    f"{integrated_supply['unified_supplyshield_risk'].mean():.2f}"
)

print(
    f"Maximum unified risk          : "
    f"{integrated_supply['unified_supplyshield_risk'].max():.2f}"
)

print(
    f"Critical records              : "
    f"{(
        integrated_supply[
            'unified_risk_band'
        ] == 'CRITICAL'
    ).sum():,}"
)

print(
    f"High-risk records             : "
    f"{(
        integrated_supply[
            'unified_risk_band'
        ] == 'HIGH'
    ).sum():,}"
)

print()
print("DATABASE")
print("-" * 100)

print(
    f"Database                     : "
    f"{DATABASE_PATH}"
)

print(
    f"Database size                : "
    f"{database_size_mb:.2f} MB"
)

print(
    f"Integrated SQL records       : "
    f"{sql_count:,}"
)

print()
print("OUTPUTS")
print("-" * 100)

print(
    f"CSV                         : "
    f"{FINAL_INTEGRATED_CSV}"
)

print(
    f"JSON                        : "
    f"{FINAL_INTEGRATED_JSON}"
)

print()
print("=" * 100)

if all_passed:

    print(
        "SUPPLYSHIELD AI INTEGRATION CHECKPOINT: PASSED"
    )

else:

    print(
        "SUPPLYSHIELD AI INTEGRATION CHECKPOINT: ATTENTION REQUIRED"
    )

print("=" * 100)


                     SUPPLYSHIELD AI
              SYSTEM INTEGRATION REPORT

INTELLIGENCE PIPELINE
----------------------------------------------------------------------------------------------------
Supply-chain intelligence      : READY
NLP / ML risk intelligence    : READY
Anomaly intelligence          : READY
WebShield intelligence        : READY
Unified risk engine           : READY
SQL persistence               : READY
Dashboard-ready dataset       : READY

DATASET
----------------------------------------------------------------------------------------------------
Integrated records            : 303
WebShield matched records     : 303
Unmatched records             : 0

RISK
----------------------------------------------------------------------------------------------------
Average unified risk          : 10.95
Maximum unified risk          : 23.38
Critical records              : 0
High-risk records             : 0

DATABASE
------------------------------------------------------

In [31]:
# ============================================================
# CELL 35 — SAFE DATABASE CHECKPOINT
# ============================================================

connection.commit()

connection.close()

print(
    "SQL transactions committed successfully."
)

print(
    "Database connection closed safely."
)

print(
    f"Database remains available at:\n{DATABASE_PATH}"
)

SQL transactions committed successfully.
Database connection closed safely.
Database remains available at:
c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\database\supplyshield.db
